# Create storm and climate datasets for each basin

Combines daily SWE, GridMET temperature and storm labels for each basin, then builds yearly climate summaries and a storm-by-storm record of snow accumulation.

**Steps**
1. **Daily climate → yearly climate** *(>128 GB, ~1 hr per basin)*
   Puts daily SWE gain (median of the ensemble), temperature and storm labels on the SWE grid with elevation bins. Saves water-year totals and means, plus a smaller file with only storm days.
2. **Pixel-storms**
   For every pixel in elevation bins 2–4 (>1500 m), records each storm's start date, number of storm days and total SWE gain.
3. **Storm groups** *(~1 min)*
   Merges storms that overlap in time into one group, then gets each group's area-weighted mean SWE gain, area and dates per elevation bin.
4. **Yearly storm summary** *(fast)*
   Per water year: total storm SWE, mean storm area, storm counts (overall and per bin) and basin-mean winter temperature.

**Inputs**
- `Data/Cleaned/<basin>/<basin>_SWE_WY*.nc4`: cleaned UCLA SWE
- `Data/DEM/<basin>_FixedElevBands.nc4`: elevation bins
- `Data/Cleaned/GridMET/temp_avg_da.nc4`, `storm_labels.nc4`
- `data_cleaning.py`: basin lists, shapes, years, `calculate_water_year`

**Outputs** (in `Data/Cleaned/`)
| File | Contents |
|------|----------|
| `<basin>_yearly_climate_ds.nc4` | Per pixel, per WY: SWE gain, storm SWE gain, mean and winter temperature |
| `<basin>_storm_days_ds.nc4` | Daily data on storm days only |
| `<basin>_storm_ds_final.nc4` | One record per pixel-storm, with its storm group |
| `<basin>_grouped_storm_ds_1.nc4` | One record per storm group and elevation bin |
| `<basin>_yearly_storm_ds_final.nc4` | Yearly storm totals and counts per basin |

Units: SWE in mm, temperature in °C.

In [ ]:
# %% Imports
import gc
import os
from contextlib import contextmanager

import dask
import numpy as np
import pandas as pd
import rioxarray  # noqa: F401  (registers the .rio accessor)
import xarray as xr
from dask.distributed import Client, LocalCluster

from config import data_folder, Basins, basin_names_list, shapes, elev_bins, years_swe, water_years_swe
from utils import calculate_water_year

cleaned = os.path.join(data_folder, 'Data/Cleaned')

In [ ]:
# %% Dask cluster for HPC
@contextmanager
def local_cluster(n_workers, total_memory_gb=384):
    """Temporary single-node Dask cluster. Closes cleanly (freeing its ports) on exit."""
    dask.config.set({
        'distributed.worker.memory.target': 0.6,
        'distributed.worker.memory.spill': 0.7,
        'distributed.worker.memory.terminate': 0.95,
    })
    with LocalCluster(n_workers=n_workers, threads_per_worker=1,
                      memory_limit=f'{total_memory_gb // n_workers}GB',
                      scheduler_port=0, dashboard_address=':0') as cluster, Client(cluster) as client:
        yield client

## set number of workers, total memory in GB
n_workers = 8
total_memory_gb = 384

In [ ]:
# %% Step 1: daily climate on the SWE grid -> yearly climate + storm-day files
# Needs >128 GB memory, ~1 hour per basin
def clip_to_grid(da, polygon, like):
    """Clip a GridMET DataArray to the basin, then nearest-neighbour regrid onto `like`."""
    da = (da.rio.write_crs('EPSG:4326')
            .rio.set_spatial_dims(x_dim='Longitude', y_dim='Latitude')
            .rio.clip([polygon], crs='EPSG:4326'))
    return (da.reindex(Day=like['Day'])
              .interp(Latitude=like['Latitude'], Longitude=like['Longitude'], method='nearest'))


def build_climate_ds(basin, polygon, temp_da, storm_labels_da):
    """Daily SWE gain [mm], temperature [˚C] and storm labels for one basin, with WY as a coordinate."""
    swe = xr.combine_by_coords([
        xr.open_dataset(os.path.join(cleaned, basin, f'{basin}_SWE_WY{y}_{y % 100 + 1}.nc4'), chunks='auto')
        for y in years_swe
    ])

    # Elevation bins on the SWE grid; 0 = no bin (int8 can't hold NaN)
    dem = xr.open_dataset(os.path.join(data_folder, f'Data/DEM/{basin}_FixedElevBands.nc4'))
    dem = dem.interp_like(swe.isel(Day=0, Stats=0), method='nearest')
    dem['ElevationBin'] = dem['ElevationBin'].fillna(0).astype('int8')

    ds = (xr.combine_by_coords([swe, dem], combine_attrs='override')
          .sel(Stats=2)  # ensemble median
          .reset_coords(['Stats', 'spatial_ref'], drop=True)
          .drop_vars('SWE_Post'))

    ds['Delta_SWE'] = ds['Delta_SWE'].where(ds['Delta_SWE'] > 0, 0) * 1000  # SWE gains only, m -> mm
    ds['Temp_Avg'] = clip_to_grid(temp_da, polygon, ds) - 273.15            # K -> ˚C
    ds['Storm_Label'] = clip_to_grid(storm_labels_da, polygon, ds)
    ds['SWE_By_Storm'] = ds['Storm_Label'].notnull() * ds['Delta_SWE']

    ds = ds.drop_vars(['crs', 'spatial_ref'], errors='ignore')
    return ds.assign_coords(WY=('Day', np.asarray(calculate_water_year(ds['Day']))))


def yearly_climate(ds):
    """Water-year totals of SWE gain, mean and winter (Dec–Mar) temperature, per pixel."""
    yearly = ds[['Delta_SWE', 'SWE_By_Storm']].groupby('WY').sum('Day')
    yearly['Temp_Avg'] = ds['Temp_Avg'].groupby('WY').mean('Day')
    yearly['ElevationBin'] = ds['ElevationBin']
    winter = ds['Day'].dt.month.isin([12, 1, 2, 3])
    yearly['Winter_Temp'] = ds['Temp_Avg'].where(winter).groupby('WY').mean('Day')

    yearly = yearly.where(yearly.WY != 2022, drop=True)
    for var in set(yearly.data_vars) - {'ElevationBin'}:
        yearly[var] = yearly[var].astype('float32')
    return yearly


temp_da = xr.open_dataarray(os.path.join(cleaned, 'GridMET/temp_avg_da.nc4'), chunks='auto')
storm_labels_da = xr.open_dataarray(os.path.join(cleaned, 'GridMET/storm_labels.nc4'), chunks='auto')

for basin, name in zip(Basins, basin_names_list):
    polygon = shapes.loc[shapes['name'] == name, 'geometry'].iloc[0]
    ds = build_climate_ds(basin, polygon, temp_da, storm_labels_da)

    path = os.path.join(cleaned, f'{basin}_yearly_climate_ds.nc4')
    yearly_climate(ds).to_netcdf(path, engine='h5netcdf')
    print(f'saved {path}')

    # Keep only days that appear in the storm labels (to shrink the file)
    storm_days = np.intersect1d(ds['Day'].values, storm_labels_da['Day'].values)
    path = os.path.join(cleaned, f'{basin}_storm_days_ds.nc4')
    ds.sel(Day=storm_days).to_netcdf(path, engine='h5netcdf')
    print(f'saved {path}')

    del ds
    gc.collect()

del temp_da, storm_labels_da
gc.collect()

In [ ]:
# %% Step 2: one record per (pixel, storm) in elevation bins 2–4
def build_storm_ds(storm_days_ds):
    """For every pixel in bins 2–4 and every storm label there: start date, days labelled, total SWE gain."""
    ds = storm_days_ds.transpose('Day', 'Latitude', 'Longitude')
    labels = ds['Storm_Label'].values
    keep = ~np.isnan(labels) & (ds['ElevationBin'].values >= 2)
    day, iy, ix = np.nonzero(keep)

    df = pd.DataFrame({
        'Latitude': ds['Latitude'].values[iy],
        'Longitude': ds['Longitude'].values[ix],
        'Label': labels[keep],
        'Day': ds['Day'].values[day],
        'Delta_SWE': ds['Delta_SWE'].values[keep],
        'Elevation': ds['Elevation'].values[iy, ix],
        'ElevationBin': ds['ElevationBin'].values[iy, ix],
    })
    storms = (df.groupby(['Latitude', 'Longitude', 'Label'])
                .agg(Start_Date=('Day', 'min'), Duration=('Day', 'size'), Delta_SWE=('Delta_SWE', 'sum'),
                     ElevationBin=('ElevationBin', 'first'), Elevation=('Elevation', 'first'))
                .reset_index())

    cols = ['Start_Date', 'Duration', 'Delta_SWE', 'ElevationBin', 'Elevation', 'Latitude', 'Longitude']
    return xr.Dataset({c: ('Storm_Label', storms[c].values) for c in cols},
                      coords={'Storm_Label': np.arange(len(storms))})


for basin in Basins:
    with local_cluster(n_workers=n_workers, total_memory_gb=total_memory_gb):
        storm_days_ds = xr.open_dataset(os.path.join(cleaned, f'{basin}_storm_days_ds.nc4'), chunks='auto').load()

    storm_ds = build_storm_ds(storm_days_ds)
    del storm_days_ds
    gc.collect()

    storm_ds.to_netcdf(os.path.join(cleaned, f'{basin}_storm_ds_1.nc4'), engine='h5netcdf')
    print(f'{basin}: {storm_ds.sizes["Storm_Label"]} pixel-storms')

In [ ]:
# %% Step 3: group storms that overlap in time; summarise each group per elevation bin (~1 min)
def label_overlapping(start, end):
    """Group IDs for date ranges that overlap, directly or through a chain of overlaps."""
    order = np.argsort(start)
    prev_max_end = np.maximum.accumulate(end[order])
    new_group = np.r_[False, start[order][1:] > prev_max_end[:-1]]
    labels = np.empty(len(start), dtype=int)
    labels[order] = np.cumsum(new_group)
    return labels


def group_storms(storm_ds):
    """Adds Group/Area/End_Date to storm_ds and returns (storm_ds, per-group-and-bin summary)."""
    start = storm_ds['Start_Date'].values
    end = start + (storm_ds['Duration'].values - 1).astype('timedelta64[D]')  # inclusive

    storm_ds = storm_ds.assign_coords(Group=('Storm_Label', label_overlapping(start, end)))
    storm_ds['Area'] = np.cos(np.deg2rad(storm_ds['Latitude'])) * (1852 / 60 * 16) ** 2  # 16-arcsec pixel [m²]
    storm_ds['End_Date'] = ('Storm_Label', end)

    by = ['Group', 'ElevationBin']
    g = storm_ds.set_coords(by)
    sums = g.assign(SWE_Area=g['Delta_SWE'] * g['Area'])[['SWE_Area', 'Area']].groupby(by).sum()
    start, end = g.groupby(by).min()['Start_Date'], g.groupby(by).max()['End_Date']

    grouped = xr.Dataset({
        'SWE': sums['SWE_Area'] / sums['Area'],  # area-weighted mean over the group's pixel-storms
        'Area': sums['Area'],
        'Start_Date': start,
        'End_Date': end,
        'Duration': (end - start).dt.days + 1,
    })
    grouped['WY'] = calculate_water_year(grouped['Start_Date'])
    grouped = grouped.sel(ElevationBin=[2, 3, 4])

    # One start/end/WY per group, across elevation bins
    grouped['Start_Date'] = grouped['Start_Date'].min('ElevationBin', skipna=True)
    grouped['End_Date'] = grouped['End_Date'].max('ElevationBin', skipna=True)
    grouped['WY'] = grouped['WY'].max('ElevationBin', skipna=True)
    return storm_ds, grouped


for basin in Basins:
    src = os.path.join(cleaned, f'{basin}_storm_ds_1.nc4')
    storm_ds, grouped = group_storms(xr.open_dataset(src))

    storm_ds.to_netcdf(os.path.join(cleaned, f'{basin}_storm_ds_final.nc4'), engine='h5netcdf')
    os.remove(src)
    grouped.to_netcdf(os.path.join(cleaned, f'{basin}_grouped_storm_ds_1.nc4'), engine='h5netcdf')

In [ ]:
# %% Step 4: yearly storm totals per basin (fast, low memory)
def yearly_storm_summary(basin):
    """Per water year: storm SWE total, mean storm area and storm counts, plus basin-mean winter temp."""
    grouped = xr.open_dataset(os.path.join(cleaned, f'{basin}_grouped_storm_ds_1.nc4')).set_coords('WY')
    by_wy = grouped.groupby('WY')
    counts = by_wy.count('Group')

    yearly = xr.combine_by_coords([by_wy.sum('Group')[['SWE']], by_wy.mean('Group')[['Area']]])
    yearly['Count_By_Bin'] = counts['SWE']      # storms reaching each elevation bin
    yearly['Count'] = counts['Start_Date']      # storms in any bin
    yearly = yearly.reindex(WY=water_years_swe)

    climate = xr.open_dataset(os.path.join(cleaned, f'{basin}_yearly_climate_ds.nc4'))
    yearly['Winter_Temp'] = climate['Winter_Temp'].mean(['Latitude', 'Longitude']) - 273.15

    yearly = yearly.rename({'Area': 'Storm_Area_Avg', 'SWE': 'SWE_By_Storm'})
    no_storm_zero = ['Count', 'Count_By_Bin', 'SWE_By_Storm', 'Storm_Area_Avg']
    yearly.update(yearly[no_storm_zero].fillna(0))  # years without storms -> 0
    return yearly


for basin in Basins:
    yearly_storm_summary(basin).to_netcdf(
        os.path.join(cleaned, f'{basin}_yearly_storm_ds_final.nc4'), engine='h5netcdf')